# RAG Evaluation Notebook

Evaluates the RAG pipeline against the oracle benchmark dataset (`RAG_test/benchmark_dataset.json`).

**Prerequisites**
- Chunked filings must be cached: run `python -m RAG_test.cache_chunked_filings` first
- `OPENAI_API_KEY` must be set in your environment

In [16]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path
REPO_ROOT = Path("__file__").resolve().parents[1] if "__file__" in dir() else Path.cwd()
while not (REPO_ROOT / "backend").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/henry/Desktop/STUDY/Y4S2/DSA4265/DSA4265-group8


In [17]:
from statistics import mean
from datetime import datetime, timezone

from RAG_test.common import (
    BENCHMARK_DATASET_PATH,
    CHUNKED_FILINGS_DIR,
    EMBEDDING_CACHE_DIR,
    RESULTS_DIR,
    chunked_filings_path,
    ensure_data_dirs,
    load_json,
    write_json,
)
from RAG_test.evaluators import (
    answer_matches_oracle,
    evaluate_oracle_match_with_llm,
    evaluate_answer_with_llm,
    optional_bertscore,
)
from backend.agents.retrieval_pipeline import (
    FilingRetrievalPipeline,
    JsonEmbeddingCache,
    OpenAIChatGenerationProvider,
    OpenAIEmbeddingProvider,
    build_chunk_records_from_prepared_filings,
)

ensure_data_dirs()
print("Imports OK")

Imports OK


## Configuration

Adjust the settings below before running the evaluation.

In [18]:
# --- Settings ---
EMBEDDING_MODEL   = "text-embedding-3-small"
GENERATION_MODEL  = "gpt-4o-mini"
USE_EMBEDDING_CACHE = True   # set False to force fresh embeddings
USE_LLM_JUDGE     = True     # set True to get relevance/completeness/faithfulness scores
TOP_K             = None     # None = use per-question-type recommended_k; or set an int e.g. 5

DATASET_PATH  = BENCHMARK_DATASET_PATH
REPORT_PATH   = RESULTS_DIR / "rag_evaluation_report.json"

## Load Dataset & Initialise Providers

In [19]:
dataset = load_json(DATASET_PATH)
print(f"Loaded {len(dataset)} benchmark examples")

from collections import Counter
type_counts = Counter(ex["question_type"] for ex in dataset)
for qtype, count in sorted(type_counts.items()):
    print(f"  {qtype}: {count}")

Loaded 30 benchmark examples
  forward_pressure: 5
  forward_revenue_driver: 5
  management_actions: 5
  profitability_margin: 5
  revenue_driver: 5
  risk_material: 5


In [20]:
embedding_cache = None
if USE_EMBEDDING_CACHE:
    cache_file = EMBEDDING_CACHE_DIR / f"{EMBEDDING_MODEL.replace('/', '__')}.json"
    embedding_cache = JsonEmbeddingCache(cache_file)
    print(f"Embedding cache: {cache_file}")

embedding_provider  = OpenAIEmbeddingProvider(model=EMBEDDING_MODEL, cache=embedding_cache)
generation_provider = OpenAIChatGenerationProvider(model=GENERATION_MODEL)
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Generation model: {GENERATION_MODEL}")

Embedding cache: /Users/henry/Desktop/STUDY/Y4S2/DSA4265/DSA4265-group8/RAG_test/data/embedding_cache/text-embedding-3-small.json
Embedding model : text-embedding-3-small
Generation model: gpt-4o-mini


## Run Evaluation

In [21]:
def select_filings(cached_payload, filing_date):
    filings = cached_payload.get("filings", [])
    if not filing_date:
        return [max(filings, key=lambda f: f.get("filing_date", ""))] if filings else []
    return [f for f in filings if f.get("filing_date") == filing_date]


pipeline_cache = {}

def get_pipeline(ticker, filing_date):
    key = (ticker, filing_date)
    if key in pipeline_cache:
        return pipeline_cache[key]
    payload  = load_json(chunked_filings_path(ticker))
    filings  = select_filings(payload, filing_date)
    if not filings:
        raise ValueError(f"No cached filings for {ticker} filing_date={filing_date!r}")
    prepared = {"ticker": ticker, "filings": filings}
    chunks   = build_chunk_records_from_prepared_filings(prepared)
    pipeline = FilingRetrievalPipeline(embedding_provider)
    pipeline.index_chunks(chunks)
    pipeline_cache[key] = (pipeline, filings)
    return pipeline_cache[key]

In [22]:
results = []

for i, example in enumerate(dataset):
    ticker       = example["ticker"]
    filing_date  = example.get("filing_date")
    cache_path   = chunked_filings_path(ticker)

    if not cache_path.exists():
        raise FileNotFoundError(
            f"Missing chunk cache for {ticker}. Run RAG_test/cache_chunked_filings.py first."
        )

    pipeline, filings = get_pipeline(ticker, filing_date)

    answer_result  = pipeline.answer_question(
        question=example["question"],
        generation_provider=generation_provider,
        k=TOP_K,
    )
    retrieved_chunks        = answer_result.get("sources", [])
    answer                  = answer_result.get("answer")
    resolved_filing_date    = filing_date or (filings[0].get("filing_date") if filings else None)

    oracle_answer_judge = evaluate_oracle_match_with_llm(
        generation_provider,
        example["question"],
        answer,
        example.get("oracle_answer"),
    )
    answer_correct = (
        oracle_answer_judge.get("correct")
        if oracle_answer_judge and oracle_answer_judge.get("correct") is not None
        else answer_matches_oracle(answer, example.get("oracle_answer"))
    )

    llm_judge = None
    if USE_LLM_JUDGE:
        llm_judge = evaluate_answer_with_llm(
            generation_provider,
            example["question"],
            answer or "",
            retrieved_chunks,
        )

    results.append({
        **example,
        "resolved_filing_date": resolved_filing_date,
        "generated_answer": answer,
        "retrieved_chunks": retrieved_chunks,
        "answer_correct": answer_correct,
        "oracle_answer_judge": oracle_answer_judge,
        "llm_judge": llm_judge,
    })

    status = "CORRECT" if answer_correct else ("WRONG" if answer_correct is False else "NO ORACLE")
    print(f"[{i+1:02d}/{len(dataset)}] {ticker} | {example['question_type']:<22} | {status}")

print("\nDone.")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[01/30] AAPL | risk_material          | CORRECT


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 500 Internal Server Error"
INFO:openai._base_client:Retrying request to /chat/completions in 0.404998 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[02/30] AAPL | forward_pressure       | CORRECT


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

[28/30] TSLA | revenue_driver         | CORRECT


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[29/30] TSLA | profitability_margin   | WRONG


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[30/30] TSLA | forward_revenue_driver | CORRECT

Done.


## Summary Metrics

In [23]:
answer_rows = [r for r in results if r["answer_correct"] is not None]
judge_rows  = [r for r in results if r.get("llm_judge")]

accuracy = mean(1.0 if r["answer_correct"] else 0.0 for r in answer_rows) if answer_rows else None

llm_overall = None
if judge_rows:
    overall_scores = [
        r["llm_judge"]["scores"]["overall"]
        for r in judge_rows
        if r["llm_judge"]["scores"]["overall"] is not None
    ]
    llm_overall = mean(overall_scores) if overall_scores else None

print(f"Examples evaluated : {len(results)}")
print(f"Answer accuracy    : {accuracy:.1%}" if accuracy is not None else "Answer accuracy    : N/A")
print(f"LLM judge overall  : {llm_overall:.2f}/5" if llm_overall is not None else "LLM judge overall  : N/A")

Examples evaluated : 30
Answer accuracy    : 66.7%
LLM judge overall  : 4.46/5


In [24]:
# Per-question-type accuracy
from collections import defaultdict

by_type = defaultdict(list)
for r in answer_rows:
    by_type[r["question_type"]].append(r["answer_correct"])

print(f"{'Question type':<25} {'Correct':>7} {'Total':>6} {'Accuracy':>9}")
print("-" * 52)
for qtype, flags in sorted(by_type.items()):
    correct = sum(flags)
    total   = len(flags)
    print(f"{qtype:<25} {correct:>7} {total:>6} {correct/total:>9.1%}")

Question type             Correct  Total  Accuracy
----------------------------------------------------
forward_pressure                4      5     80.0%
forward_revenue_driver          5      5    100.0%
management_actions              1      5     20.0%
profitability_margin            3      5     60.0%
revenue_driver                  5      5    100.0%
risk_material                   2      5     40.0%


In [25]:
# Per-company accuracy
by_company = defaultdict(list)
for r in answer_rows:
    by_company[r["ticker"]].append(r["answer_correct"])

print(f"{'Ticker':<8} {'Correct':>7} {'Total':>6} {'Accuracy':>9}")
print("-" * 35)
for ticker, flags in sorted(by_company.items()):
    correct = sum(flags)
    total   = len(flags)
    print(f"{ticker:<8} {correct:>7} {total:>6} {correct/total:>9.1%}")

Ticker   Correct  Total  Accuracy
-----------------------------------
AAPL           6      6    100.0%
GOOG           4      6     66.7%
META           2      6     33.3%
NVDA           4      6     66.7%
TSLA           4      6     66.7%


## Inspect Individual Results

In [26]:
# Show all incorrect answers
wrong = [r for r in results if r["answer_correct"] is False]
print(f"{len(wrong)} incorrect answers\n")

for r in wrong:
    print(f"{'='*80}")
    print(f"Ticker       : {r['ticker']}  |  {r['question_type']}")
    print(f"Question     : {r['question']}")
    print(f"Oracle       : {r.get('oracle_answer', 'N/A')}")
    print(f"Generated    : {r.get('generated_answer', 'N/A')}")
    if r.get("oracle_answer_judge"):
        print(f"Judge reason : {r['oracle_answer_judge'].get('reason', '')}")

10 incorrect answers

Ticker       : GOOG  |  risk_material
Question     : What are the most material risks explicitly disclosed by management this quarter?
Oracle       : Management did not identify a broad reset to risk factors; it pointed back to the 2024 10-K and ongoing legal proceedings. The most material quarter-specific disclosed issue was the EC's September 5, 2025 competition decision and $3.5B fine, alongside continuing global antitrust and competition investigations and litigation.
Generated    : The most material risks explicitly disclosed by management this quarter include various uncertainties related to their operations and financial results, as described in Part I, Item 1A, "Risk Factors" in their Annual Report on Form 10-K for the year ended December 31, 2024. Specific risks mentioned include potential harm to their business, reputation, financial condition, and operating results, which could also affect the trading price of their Class A and Class C stock. Additional

In [27]:
# Show LLM judge scores breakdown (if enabled)
if judge_rows:
    print(f"{'Ticker':<6} {'Type':<22} {'Rel':>4} {'Comp':>5} {'Faith':>6} {'Overall':>8}")
    print("-" * 55)
    for r in results:
        if not r.get("llm_judge"):
            continue
        sc = r["llm_judge"]["scores"]
        print(
            f"{r['ticker']:<6} {r['question_type']:<22} "
            f"{str(sc.get('relevance','?')):>4} "
            f"{str(sc.get('completeness','?')):>5} "
            f"{str(sc.get('faithfulness','?')):>6} "
            f"{str(sc.get('overall','?')):>8}"
        )
else:
    print("LLM judge was disabled. Set USE_LLM_JUDGE = True and re-run.")

Ticker Type                    Rel  Comp  Faith  Overall
-------------------------------------------------------
AAPL   risk_material             5     3      4        4
AAPL   forward_pressure          5     4      5     None
AAPL   management_actions        4     2      3        3
AAPL   revenue_driver            5     5      5        5
AAPL   profitability_margin      5     4      5     None
AAPL   forward_revenue_driver    5     4      5     None
GOOG   risk_material             5     4      5     None
GOOG   forward_pressure          5     5      5        5
GOOG   management_actions        5     2      4        3
GOOG   revenue_driver            5     5      5        5
GOOG   profitability_margin      5     4      5     None
GOOG   forward_revenue_driver    5     4      5     None
META   risk_material             5     5      5        5
META   forward_pressure          5     5      5        5
META   management_actions        5     4      5     None
META   revenue_driver           

## Reranker Comparison

Run the same evaluation with the cross-encoder reranker enabled, then compare accuracy and LLM judge scores against the baseline (no reranker).

> **Note:** Requires `sentence-transformers` installed: `pip install sentence-transformers`

In [28]:
from backend.agents.retrieval_pipeline import CrossEncoderReranker

reranker = CrossEncoderReranker(model_name="BAAI/bge-reranker-large")

pipeline_cache_reranker = {}

def get_pipeline_with_reranker(ticker, filing_date):
    key = (ticker, filing_date)
    if key in pipeline_cache_reranker:
        return pipeline_cache_reranker[key]
    payload  = load_json(chunked_filings_path(ticker))
    filings  = select_filings(payload, filing_date)
    if not filings:
        raise ValueError(f"No cached filings for {ticker} filing_date={filing_date!r}")
    prepared = {"ticker": ticker, "filings": filings}
    chunks   = build_chunk_records_from_prepared_filings(prepared)
    pipeline = FilingRetrievalPipeline(embedding_provider, reranker=reranker)
    pipeline.index_chunks(chunks)
    pipeline_cache_reranker[key] = (pipeline, filings)
    return pipeline_cache_reranker[key]

results_reranker = []

for i, example in enumerate(dataset):
    ticker      = example["ticker"]
    filing_date = example.get("filing_date")

    pipeline, filings = get_pipeline_with_reranker(ticker, filing_date)

    answer_result = pipeline.answer_question(
        question=example["question"],
        generation_provider=generation_provider,
        k=TOP_K,
    )
    answer          = answer_result.get("answer")
    retrieved_chunks = answer_result.get("sources", [])

    oracle_judge = evaluate_oracle_match_with_llm(
        generation_provider,
        example["question"],
        answer,
        example.get("oracle_answer"),
    )
    answer_correct = (
        oracle_judge.get("correct")
        if oracle_judge and oracle_judge.get("correct") is not None
        else answer_matches_oracle(answer, example.get("oracle_answer"))
    )

    llm_judge = evaluate_answer_with_llm(
        generation_provider,
        example["question"],
        answer or "",
        retrieved_chunks,
    ) if USE_LLM_JUDGE else None

    results_reranker.append({
        **example,
        "generated_answer": answer,
        "answer_correct": answer_correct,
        "llm_judge": llm_judge,
    })

    status = "CORRECT" if answer_correct else ("WRONG" if answer_correct is False else "NO ORACLE")
    print(f"[{i+1:02d}/{len(dataset)}] {ticker} | {example['question_type']:<22} | {status}")

print("\nDone.")

INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: mps
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.71s/it]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[01/30] AAPL | risk_material          | WRONG


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[02/30] AAPL | forward_pressure       | CORRECT


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

[28/30] TSLA | revenue_driver         | CORRECT


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[29/30] TSLA | profitability_margin   | WRONG


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.87s/it]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[30/30] TSLA | forward_revenue_driver | CORRECT

Done.


In [33]:
# --- Side-by-side comparison: no reranker vs reranker ---
from collections import defaultdict

def summarise(res):
    rows = [r for r in res if r["answer_correct"] is not None]
    acc  = mean(1.0 if r["answer_correct"] else 0.0 for r in rows) if rows else None
    judge_rows = [r for r in res if r.get("llm_judge")]
    overall_scores = [
        r["llm_judge"]["scores"]["overall"]
        for r in judge_rows
        if r["llm_judge"]["scores"]["overall"] is not None
    ]
    llm_overall = mean(overall_scores) if overall_scores else None
    by_type = defaultdict(list)
    for r in rows:
        by_type[r["question_type"]].append(r["answer_correct"])
    return acc, llm_overall, rows, by_type

acc_base,     llm_base,     rows_base,     by_type_base     = summarise(results)
acc_reranker, llm_reranker, rows_reranker, by_type_reranker = summarise(results_reranker)

# BERTScore (optional — requires bert-score package)
bert_base     = optional_bertscore(
    [r.get("generated_answer") or "" for r in rows_base],
    [r.get("oracle_answer") or "" for r in rows_base],
)
bert_reranker = optional_bertscore(
    [r.get("generated_answer") or "" for r in rows_reranker],
    [r.get("oracle_answer") or "" for r in rows_reranker],
)
bert_base_mean     = mean(bert_base)     if bert_base     else None
bert_reranker_mean = mean(bert_reranker) if bert_reranker else None

print(f"{'Metric':<30} {'No Reranker':>12} {'With Reranker':>14} {'Delta':>8}")
print("-" * 68)
print(f"{'Answer accuracy':<30} {acc_base:>11.1%} {acc_reranker:>13.1%} {acc_reranker - acc_base:>+8.1%}")
print(f"{'LLM judge overall (/5)':<30} {llm_base:>11.2f} {llm_reranker:>13.2f} {llm_reranker - llm_base:>+8.2f}")
if bert_base_mean is not None and bert_reranker_mean is not None:
    print(f"{'BERTScore F1':<30} {bert_base_mean:>11.4f} {bert_reranker_mean:>13.4f} {bert_reranker_mean - bert_base_mean:>+8.4f}")
else:
    print(f"{'BERTScore F1':<30} {'N/A':>12} {'N/A':>14}   (install bert-score to enable)")
print()
print(f"{'Question type':<25} {'No Reranker':>12} {'With Reranker':>14} {'Delta':>8}")
print("-" * 62)
all_types = sorted(set(by_type_base) | set(by_type_reranker))
for qtype in all_types:
    b = by_type_base.get(qtype, [])
    r = by_type_reranker.get(qtype, [])
    acc_b = mean(b) if b else 0.0
    acc_r = mean(r) if r else 0.0
    delta = acc_r - acc_b
    flag  = " <<< WORSE" if delta < -0.1 else (" >>> BETTER" if delta > 0.1 else "")
    print(f"{qtype:<25} {acc_b:>11.1%} {acc_r:>13.1%} {delta:>+8.1%}{flag}")

winner = "No Reranker" if acc_base >= acc_reranker else "With Reranker"
print(f"\nVerdict: {winner} wins overall.")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Metric                          No Reranker  With Reranker    Delta
--------------------------------------------------------------------
Answer accuracy                      66.7%         60.0%    -6.7%
LLM judge overall (/5)                4.46          4.25    -0.21
BERTScore F1                        0.8466        0.8422  -0.0044

Question type              No Reranker  With Reranker    Delta
--------------------------------------------------------------
forward_pressure                80.0%         60.0%   -20.0% <<< WORSE
forward_revenue_driver         100.0%        100.0%    +0.0%
management_actions              20.0%         20.0%    +0.0%
profitability_margin            60.0%         60.0%    +0.0%
revenue_driver                 100.0%        100.0%    +0.0%
risk_material                   40.0%         20.0%   -20.0% <<< WORSE

Verdict: No Reranker wins overall.


## Save Report

In [34]:
# Optional BERTScore (requires bert-score package)
pipeline_bertscores = optional_bertscore(
    [r.get("generated_answer") or "" for r in answer_rows],
    [r.get("oracle_answer") or "" for r in answer_rows],
)
if pipeline_bertscores:
    for row, score in zip(answer_rows, pipeline_bertscores):
        row["bertscore_f1"] = score
    print(f"BERTScore F1 (mean): {mean(pipeline_bertscores):.4f}")
else:
    print("BERTScore not computed (install bert-score to enable)")

report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(DATASET_PATH),
    "chunk_cache_dir": str(CHUNKED_FILINGS_DIR),
    "embedding_model": EMBEDDING_MODEL,
    "generation_model": GENERATION_MODEL,
    "k": TOP_K if TOP_K is not None else "per_question_type",
    "summary": {
        "num_examples": len(results),
        "answer_accuracy": accuracy,
        "pipeline_bertscore_f1": mean(pipeline_bertscores) if pipeline_bertscores else None,
        "llm_judge_overall": llm_overall,
    },
    "results": results,
}

write_json(REPORT_PATH, report)
print(f"Report saved to {REPORT_PATH}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1 (mean): 0.8466
Report saved to /Users/henry/Desktop/STUDY/Y4S2/DSA4265/DSA4265-group8/RAG_test/results/rag_evaluation_report.json
